In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import os


class MLP(nn.Module):
    def __init__(self, num_classes):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc_stack = nn.Sequential(
            nn.Linear(3 * 224 * 224, 256), 
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.fc_stack(x)
        return logits

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load('best_model.pth', map_location=device)


best_model_name = checkpoint['model_name']
num_classes = checkpoint['num_classes']
class_names = checkpoint['class_names']


if best_model_name == 'CNN':
    model = SimpleCNN(num_classes=num_classes)
else:
    model = MLP(num_classes=num_classes)


model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"Model '{best_model_name}' loaded successfully with {num_classes} classes.")

Model 'CNN' loaded successfully with 38 classes.


In [2]:

inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict(image_path):
    
    image = Image.open(image_path).convert('RGB')
    image_tensor = inference_transforms(image).unsqueeze(0).to(device)
    
   
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
        
    class_idx = predicted.item()
    return class_names[class_idx], confidence.item()




PREDICTION

In [4]:
test_img = r"C:\Users\alann\OneDrive\Desktop\plantdisease\dataset\images\val\Tomato___healthy\0cb10f98-491d-4e1f-b8ea-4fb0f1b3675f___GH_HL Leaf 333.JPG"  # Replace with your actual image path
if os.path.exists(test_img):
    label, prob = predict(test_img)
    print(f"Prediction: {label} ({prob:.2%})")
else:
    print("Please check the image path.")

Prediction: Tomato___healthy (100.00%)
